In [2]:
# ============================================================
# PLOTLY INTERACTIVE CHARTS
# Part 1
# ============================================================

import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# --------------------------------------------------------
# Paths
# --------------------------------------------------------

DATA="../data/processed/Combined_Data.xlsx"

OUTPUT=Path("../output/plotly")

OUTPUT.mkdir(parents=True,exist_ok=True)

# --------------------------------------------------------
# Load Data
# --------------------------------------------------------

df=pd.read_excel(
    DATA,
    sheet_name="Support Tickets"
)

# --------------------------------------------------------
# Datetime
# --------------------------------------------------------

df["Created Time (Ticket)"]=pd.to_datetime(
    df["Created Time (Ticket)"],
    dayfirst=True,
    errors="coerce"
)

df["Ticket Closed Time"]=pd.to_datetime(
    df["Ticket Closed Time"],
    dayfirst=True,
    errors="coerce"
)

df["First Response Time"]=pd.to_datetime(
    df["First Response Time"],
    dayfirst=True,
    errors="coerce"
)

# --------------------------------------------------------
# Feature Engineering
# --------------------------------------------------------

df["Resolution Hours"]=(
    df["Ticket Closed Time"]-
    df["Created Time (Ticket)"]
).dt.total_seconds()/3600

df["First Response Hours"]=(
    df["First Response Time"]-
    df["Created Time (Ticket)"]
).dt.total_seconds()/3600

print(df.head())

   Ticket Id          Student or WP        Program Name Status (Ticket)  \
0       6403  Working Professionals   Fullstack Program          Closed   
1       6415  Working Professionals     Backend Program       Duplicate   
2       6420  Working Professionals   Fullstack Program          Closed   
3       6402                Student   Fullstack Program          Closed   
4       6423  Working Professionals  Fellowship Program          Closed   

  Created Time (Ticket)  Ticket Closed Time First Response Time  \
0   2021-05-14 01:09:00 2021-05-14 19:04:00 2021-05-14 08:51:00   
1   2021-05-14 10:12:00 2021-05-14 11:23:00                 NaT   
2   2021-05-14 11:46:00 2021-05-16 00:09:00 2021-05-14 18:14:00   
3   2021-05-14 01:08:00 2021-05-14 19:04:00 2021-05-14 14:45:00   
4   2021-05-14 12:17:00 2021-05-14 20:39:00 2021-05-14 12:21:00   

       Project Phase  Resolution Hours  First Response Hours  
0        trial phase         17.916667              7.700000  
1        trial phase

In [3]:
daily=(
    df.groupby(
        df["Created Time (Ticket)"].dt.date
    )
    .size()
    .reset_index(name="Tickets")
)

fig1=go.Figure()

fig1.add_trace(

    go.Scatter(

        x=daily["Created Time (Ticket)"],

        y=daily["Tickets"],

        mode="lines+markers",

        line=dict(
            color="#1f77b4",
            width=2
        ),

        marker=dict(size=6),

        hovertemplate=

        "<b>Date:</b> %{x}<br>"+

        "Tickets: %{y}<extra></extra>"

    )

)

fig1.update_layout(

    title="Daily Ticket Trend",

    xaxis_title="Date",

    yaxis_title="Tickets",

    hovermode="x unified",

    height=550

)

fig1.write_html(

    OUTPUT/"chart1_ticket_trend.html"

)

fig1.show()

In [4]:
program=(

df.groupby("Program Name")

.agg(

Tickets=("Ticket Id","count"),

Avg_Resolution=("Resolution Hours","mean"),

Avg_Response=("First Response Hours","mean")

)

.reset_index()

)

fig2=go.Figure()

fig2.add_trace(

go.Bar(

x=program["Program Name"],

y=program["Tickets"],

customdata=program[

["Avg_Resolution",

"Avg_Response"]

],

hovertemplate=

"<b>%{x}</b><br>"+

"Tickets: %{y}<br>"+

"Avg Resolution: %{customdata[0]:.2f} hrs<br>"+

"Avg Response: %{customdata[1]:.2f} hrs<extra></extra>",

marker_color="#ff7f0e"

)

)

fig2.update_layout(

title="Program Performance",

xaxis_title="Program",

yaxis_title="Tickets",

height=550

)

fig2.write_html(

OUTPUT/"chart2_program_performance.html"

)

fig2.show()

In [7]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# -------------------------------------------------------
# Load Dataset
# -------------------------------------------------------

DATA = Path("processed/Combined_Data.xlsx")

if not DATA.exists():
    DATA = Path("../data/processed/Combined_Data.xlsx")

df = pd.read_excel(DATA, sheet_name="Support Tickets")

# -------------------------------------------------------
# Clean Data
# -------------------------------------------------------

df["Created Time (Ticket)"] = pd.to_datetime(
    df["Created Time (Ticket)"],
    dayfirst=True,
    errors="coerce"
)

df["Month"] = df["Created Time (Ticket)"].dt.to_period("M").astype(str)

# -------------------------------------------------------
# Metric 1 : Tickets Created
# -------------------------------------------------------

tickets_created = (
    df.groupby("Month")
      .size()
      .reset_index(name="Count")
)

# -------------------------------------------------------
# Metric 2 : Closed Tickets
# -------------------------------------------------------

closed = (
    df[df["Status (Ticket)"].str.contains("Closed", case=False, na=False)]
    .groupby("Month")
    .size()
    .reset_index(name="Count")
)

# -------------------------------------------------------
# Metric 3 : Open Tickets
# -------------------------------------------------------

open_ticket = (
    df[df["Status (Ticket)"].str.contains("Open", case=False, na=False)]
    .groupby("Month")
    .size()
    .reset_index(name="Count")
)

# Make all months common
months = sorted(df["Month"].dropna().unique())

created_map = tickets_created.set_index("Month")["Count"].to_dict()
closed_map = closed.set_index("Month")["Count"].to_dict()
open_map = open_ticket.set_index("Month")["Count"].to_dict()

created_values = [created_map.get(m,0) for m in months]
closed_values = [closed_map.get(m,0) for m in months]
open_values = [open_map.get(m,0) for m in months]

# -------------------------------------------------------
# Plotly Figure
# -------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=months,
        y=created_values,
        name="Tickets Created",
        visible=True,
        marker_color="royalblue"
    )
)

fig.add_trace(
    go.Bar(
        x=months,
        y=closed_values,
        name="Closed Tickets",
        visible=False,
        marker_color="green"
    )
)

fig.add_trace(
    go.Bar(
        x=months,
        y=open_values,
        name="Open Tickets",
        visible=False,
        marker_color="red"
    )
)

# -------------------------------------------------------
# Dropdown
# -------------------------------------------------------

fig.update_layout(

    updatemenus=[
        dict(

            buttons=[

                dict(
                    label="Tickets Created",
                    method="update",
                    args=[
                        {"visible":[True,False,False]},
                        {"title":"Monthly Tickets Created"}
                    ]
                ),

                dict(
                    label="Closed Tickets",
                    method="update",
                    args=[
                        {"visible":[False,True,False]},
                        {"title":"Monthly Closed Tickets"}
                    ]
                ),

                dict(
                    label="Open Tickets",
                    method="update",
                    args=[
                        {"visible":[False,False,True]},
                        {"title":"Monthly Open Tickets"}
                    ]
                )

            ],

            direction="down",
            showactive=True,
            x=0,
            y=1.15

        )
    ],

    title="Monthly Ticket Dashboard",
    xaxis_title="Month",
    yaxis_title="Number of Tickets",
    template="plotly_white",
    height=600
)

fig.write_html("../output/chart3_dropdown_dashboard.html")

fig.show()

print("Saved -> ../output/chart3_dropdown_dashboard.html")

Saved -> ../output/chart3_dropdown_dashboard.html


In [13]:
# ==========================================================
# TASK 3 - Enable Zoom, Pan & Reset (Plotly Interactive)
# ==========================================================

import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# ----------------------------------------------------------
# File Paths
# ----------------------------------------------------------

BASE_DIR = Path.cwd().parent      # SW2627-Python-Nexus

DATA_FILE = BASE_DIR / "data" / "raw" / "Combined_Data.xlsx"

OUTPUT_DIR = BASE_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

# ----------------------------------------------------------
# Load Dataset
# ----------------------------------------------------------

df = pd.read_excel(
    DATA_FILE,
    sheet_name="Support Tickets"
)

# ----------------------------------------------------------
# Convert Date
# ----------------------------------------------------------

df["Created Time (Ticket)"] = pd.to_datetime(
    df["Created Time (Ticket)"],
    dayfirst=True,
    errors="coerce"
)

df = df.dropna(subset=["Created Time (Ticket)"])

# ----------------------------------------------------------
# Daily Ticket Count
# ----------------------------------------------------------

daily = (
    df.groupby(df["Created Time (Ticket)"].dt.date)
      .size()
      .reset_index(name="Tickets")
)

daily["Created Time (Ticket)"] = pd.to_datetime(
    daily["Created Time (Ticket)"]
)

# ----------------------------------------------------------
# Interactive Plotly Chart
# ----------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=daily["Created Time (Ticket)"],
        y=daily["Tickets"],
        mode="lines+markers",
        marker=dict(size=7),
        line=dict(width=2, color="royalblue"),
        hovertemplate=
        "<b>Date:</b> %{x|%d-%b-%Y}<br>"
        "<b>Tickets:</b> %{y}<extra></extra>"
    )
)

# ----------------------------------------------------------
# Layout
# ----------------------------------------------------------

fig.update_layout(

    title="Interactive Ticket Trend",

    xaxis_title="Date",

    yaxis_title="Number of Tickets",

    hovermode="closest",

    dragmode="zoom",

    template="plotly_white",

    height=600
)

# ----------------------------------------------------------
# Enable Range Slider
# ----------------------------------------------------------

fig.update_xaxes(

    rangeslider_visible=True,

    showspikes=True,

    spikecolor="black",

    spikemode="across"

)

# ----------------------------------------------------------
# Save HTML
# ----------------------------------------------------------

fig.write_html(
    OUTPUT_DIR / "chart4_interactive.html"
)

print("Saved Successfully")
print(OUTPUT_DIR / "chart4_interactive.html")

Saved Successfully
c:\Users\NallapureddyRishitha\OneDrive\Desktop\simulation\SW2627-Python-Nexus\output\chart4_interactive.html


In [14]:
from pathlib import Path

for file in Path.cwd().rglob("*.xlsx"):
    print(file)